# Занятие 2. Веб-сбор, источники и правовая рамка

**Цель практики:** собрать маленькую таблицу источников через открытые API, извлечь текст, проверить шум и зафиксировать лицензионные/этические риски.

Работает в бесплатном Colab на CPU. Используем Wikipedia API вместо агрессивного скрейпинга сайтов.

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

## 1. Получаем страницы из Удмуртской Википедии

In [ ]:
LANG = 'udm'
API = f'https://{LANG}.wikipedia.org/w/api.php'

def search_pages(query, limit=10):
    params = {
        'action': 'query',
        'list': 'search',
        'srsearch': query,
        'srlimit': limit,
        'format': 'json',
    }
    r = requests.get(API, params=params, timeout=30)
    r.raise_for_status()
    return r.json()['query']['search']

hits = search_pages('удмурт', 10)
sources = pd.DataFrame([{
    'title': h['title'],
    'pageid': h['pageid'],
    'url': f'https://{LANG}.wikipedia.org/wiki/' + requests.utils.quote(h['title'].replace(' ', '_')),
    'type': 'wiki_article',
    'license_or_access': 'Wikipedia text: CC BY-SA, check page history and terms',
    'notes': 'API search result',
} for h in hits])
show_df(sources, 10)
save_artifact('lesson02_sources.csv', sources)

## 2. Извлекаем тексты и считаем базовый шум

In [ ]:
def get_extract(pageid):
    params = {
        'action': 'query',
        'pageids': pageid,
        'prop': 'extracts|info',
        'explaintext': 1,
        'inprop': 'url',
        'format': 'json',
    }
    r = requests.get(API, params=params, timeout=30)
    r.raise_for_status()
    page = next(iter(r.json()['query']['pages'].values()))
    return page.get('extract', ''), page.get('fullurl')

texts = []
for _, row in sources.iterrows():
    txt, url = get_extract(row.pageid)
    texts.append({'pageid': row.pageid, 'title': row.title, 'url': url, 'text': txt, 'chars': len(txt)})

corpus = pd.DataFrame(texts)
corpus['cyrillic_share'] = corpus['text'].apply(lambda x: len(re.findall(r'[А-Яа-яЁёӐ-ӿ]', x)) / max(len(x), 1))
corpus['line_count'] = corpus['text'].apply(lambda x: len([l for l in x.splitlines() if l.strip()]))
show_df(corpus[['title', 'chars', 'cyrillic_share', 'line_count', 'url']], 10)
save_artifact('lesson02_raw_corpus.csv', corpus)

## 3. Простая фильтрация и отчет агента-ревьюера

In [ ]:
filtered = corpus[(corpus['chars'] >= 200) & (corpus['cyrillic_share'] > 0.45)].copy()
filtered['duplicate_text'] = filtered.duplicated('text')

review = {
    'n_sources': len(sources),
    'n_texts_kept': len(filtered),
    'too_short': int((corpus['chars'] < 200).sum()),
    'low_cyrillic_share': int((corpus['cyrillic_share'] <= 0.45).sum()),
    'duplicates': int(filtered['duplicate_text'].sum()),
    'human_checks': [
        'проверить лицензию и условия использования',
        'проверить, что текст действительно на нужном языке',
        'убрать навигацию, списки и служебные фрагменты',
    ],
}
print(json.dumps(review, ensure_ascii=False, indent=2))
save_artifact('lesson02_filtered_corpus.csv', filtered)
save_artifact('lesson02_review.json', json.dumps(review, ensure_ascii=False, indent=2))

## Вопросы для отчёта

1. Какие страницы пришлось бы исключить и почему?
2. Достаточно ли API-источника для курса или нужны другие сайты/архивы?
3. Какие поля метаданных нужно добавить в `sources.csv`?
4. Что агент может сделать сам, а что должен подтвердить человек?